# 04 · Curation — a LightRAG knowledge graph is maintainable

Merge duplicate entities, edit entities/relations, and delete by entity or document — on a small, deterministic `lightrag_curation` graph built from a few crafted paragraphs.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
from _common.rag import reset_rag
DOCS = [
    "OpenAI is an AI research company in San Francisco. Sam Altman is the CEO of OpenAI. OpenAI created GPT-4.",
    "Microsoft is a technology company in Redmond. Satya Nadella leads Microsoft. Microsoft invested in OpenAI.",
    "Open AI partnered with Microsoft to deploy models on the Azure cloud platform, bringing GPT-4 to Azure.",
    "Sam Altman was president of Y Combinator before OpenAI. He is a prominent figure in Silicon Valley.",
    "Azure is Microsoft's cloud platform. It hosts large language models including GPT-4 for enterprises.",
    "Google DeepMind is a rival AI lab competing with OpenAI on language models.",
]
rag = build_rag("lightrag_curation")
await rag.initialize_storages(); await initialize_pipeline_status()
g = rag.chunk_entity_relation_graph
await reset_rag(rag)
await rag.ainsert(DOCS, ids=[f"doc-{i}" for i in range(len(DOCS))],
                  file_paths=[f"crafted-{i}" for i in range(len(DOCS))])
labels = await g.get_all_labels()
print(len(labels), "entities:", ", ".join(labels))

12 entities: Azure, GPT-4, Google DeepMind, Microsoft, Open AI, OpenAI, Redmond, Sam Altman, San Francisco, Satya Nadella, Silicon Valley, Y Combinator


## Merge duplicate entities (`amerge_entities`)

The extractor kept *Open AI* and *OpenAI* separate — fold one into the other.

In [2]:
src, tgt = "Open AI", "OpenAI"
if src in labels and tgt in labels:
    await rag.amerge_entities(source_entities=[src], target_entity=tgt)
    print(f"'{src}' present after merge: {await g.has_node(src)}; entities now: {len(await g.get_all_labels())}")
else:
    print("(extractor already merged them)")

'Open AI' present after merge: False; entities now: 11


## Edit an entity and a relation (`aedit_entity`, `aedit_relation`)

In [3]:
await rag.aedit_entity("OpenAI", {"description": "OpenAI (curated).", "entity_type": "organization"})
node = await g.get_node("OpenAI")
print("entity:", node.get("entity_type"), "-", (node.get("description") or "")[:60])
edges = await g.get_node_edges("OpenAI")
if edges:
    s, t = edges[0]
    await rag.aedit_relation(s, t, {"description": "curated relationship", "weight": 9.0})
    print("relation", s, "->", t, "weight:", (await g.get_edge(s, t)).get("weight"))

entity: organization - OpenAI (curated).


relation OpenAI -> Azure weight: 9.0


## Delete an entity, then a whole document

In [4]:
leaf = (await g.get_all_labels())[-1]
await rag.adelete_by_entity(leaf)
print(f"deleted entity '{leaf}'; present: {await g.has_node(leaf)}; entities: {len(await g.get_all_labels())}")
rows, total = await rag.doc_status.get_docs_paginated(page=1, page_size=1)
res = await rag.adelete_by_doc_id(rows[0][0])
print(f"deleted document {rows[0][0]}: {getattr(res,'status',res)}; "
      f"documents now: {(await rag.doc_status.get_all_status_counts()).get('all')}")

deleted entity 'Y Combinator'; present: False; entities: 10


deleted document doc-2: success; documents now: 5


In [5]:
await rag.finalize_storages()